In [1]:
import glob
import pandas as pd

In [2]:
RESULT_DIR = 'aggregated/'
DATA_DIR = 'sentimental_flair/'

In [3]:
# https://www.geeksforgeeks.org/getting-all-csv-files-from-a-directory-using-python/
# csv files in the path 
files = glob.glob(DATA_DIR + "/*.csv") 
  
# defining an empty list to store  
# content 
df = pd.DataFrame() 
content = [] 
  
# checking all the csv files in the  
# specified path 
for filename in files: 
    
    # reading content of csv file 
    # content.append(filename) 
    df = pd.read_csv(filename, index_col=None) 
    content.append(df) 
  
# converting content to data frame 
df = pd.concat(content) 
df.reset_index(inplace=True)
print(df) 

       index                                           Headline  \
0          0  Wild Swings in Money-Market Rates Highlight Li...   
1          1      More Than 20 Are Hurt in Bronx Apartment Fire   
2          2                  Games Growth Not a Given in China   
3          3      Stricter Bank Supervision Didn’t Hurt Lending   
4          4                               A Moment of Contempt   
...      ...                                                ...   
13043   4081  Richard Rogers, Architect Behind Landmark Pomp...   
13044   4082  Johnny Isakson, 76, Longtime Senator From Geor...   
13045   4083  Lucía Hiriart, Powerful Wife of Chile’s Dictat...   
13046   4084               Would You Sponsor an Afghan Refugee?   
13047   4085  It’s Been a Hard Year. These 3 Charities Could...   

                        Source  Year  Month  Day  Date_id  Pre-Covid  Bias  \
0      The Wall Street Journal  2018      1    2        0          0     1   
1      The Wall Street Journal  2018   

In [4]:
result_df = pd.DataFrame(columns=['date','negative_l','total_l','negative_r','total_r'])
result_df = pd.DataFrame({
                        'date': pd.Series(dtype='str'),
                        'pre_covid': pd.Series(dtype='int'),
                        'negative_l': pd.Series(dtype='int'),
                        'total_l': pd.Series(dtype='int'),
                        'negative_r': pd.Series(dtype='int'),
                        'total_r': pd.Series(dtype='int'),
                        })
result_df

,date,pre_covid,negative_l,total_l,negative_r,total_r


In [5]:
for ind in df.index:
    date = str(df['Year'][ind]) + '-' + str(df['Month'][ind]) + '-' + str(df['Day'][ind])
    pre_covid = df['Pre-Covid'][ind]
    bias = df['Bias'][ind]
    sentiment = df['Sentiment'][ind]

    selected_df = result_df[result_df['date'] == date]
    if selected_df.empty: # new date for result_df
        row_data = []
        row_data.append(date) # date_id
        row_data.append(pre_covid) # pre_covid
        if bias == 0: # left
            if sentiment == -1:
                row_data.append(1) # negative_l : negative
                row_data.append(1) # total_l
                row_data.append(0) # negative_r
                row_data.append(0) # total_r
            else:
                row_data.append(0) # negative_l : neutral or positive
                row_data.append(1) # total_l
                row_data.append(0) # negative_r
                row_data.append(0) # total_r
        elif bias == 1: # right
            if sentiment == -1:
                row_data.append(0) # negative_l
                row_data.append(0) # total_l
                row_data.append(1) # negative_r : negative
                row_data.append(1) # total_r
            else:
                row_data.append(0) # negative_l
                row_data.append(0) # total_l
                row_data.append(0) # negative_r : neutral or positive
                row_data.append(1) # total_r
        row = pd.Series(row_data, index=result_df.columns)
        result_df = pd.concat([result_df, pd.DataFrame([row])], ignore_index=True)
    else: # date exists
        row_idx = result_df.index[result_df['date'] == date].tolist()[0]
        if bias == 0: # left
            if sentiment == -1:
                result_df.loc[row_idx, ['negative_l']] = result_df.loc[row_idx, ['negative_l']] + 1
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
            else:
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
        elif bias == 1: # right
            if sentiment == -1:
                result_df.loc[row_idx, ['negative_r']] = result_df.loc[row_idx, ['negative_r']] + 1
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
            else:
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
    

In [6]:
date_id = result_df.index
result_df.insert(0, 'date_id', date_id)
result_df

,date_id,date,pre_covid,negative_l,total_l,negative_r,total_r
0,0,2018-1-2,0,22,43,39,74
1,1,2018-1-9,0,24,41,45,88
2,2,2018-2-8,0,30,46,51,93
3,3,2018-3-14,0,26,40,58,98
4,4,2018-4-10,0,29,42,53,100
...,...,...,...,...,...,...,...
95,95,2021-9-18,1,26,44,14,29
96,96,2021-10-10,1,19,56,26,50
97,97,2021-11-5,1,25,48,50,96
98,98,2021-11-16,1,27,47,49,93


In [7]:
result_file = RESULT_DIR + 'aggregated_full.csv'
result_df.to_csv(result_file, index=False)